# 01 — Register agents & load ground truth

Registers the data agents to evaluate and loads ground-truth question banks from `Files/ground_truth/*.csv`.

- **Auto-discover**: registers every data agent in this workspace (bank = `DEFAULT_BANK`).
- **EXTRA_AGENTS**: explicit entries, including agents in *other* workspaces.
- Bank CSVs need columns `question,expected_answer` (optional `category,tags`). File name = bank name.

Re-running is safe: agents are merged (existing rows win), banks are reloaded from files.

In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "{{DEFAULT_LAKEHOUSE_NAME}}",
        "id": "{{DEFAULT_LAKEHOUSE_ID}}",
        "workspaceId": "{{WORKSPACE_ID}}"
    }
}

In [ ]:
# PARAMETERS -----------------------------------------------------------------
AUTO_DISCOVER = True      # register all data agents found in this workspace
DEFAULT_BANK = "default"  # bank assigned to newly discovered agents
EXTRA_AGENTS = []         # list of dicts or a JSON string, e.g.
# [{"agent_name": "MyAgent", "workspace_name": "Other WS",
#   "stage": "production", "question_bank": "my_bank", "enabled": True}]

In [ ]:
import json
if isinstance(EXTRA_AGENTS, str):
    EXTRA_AGENTS = json.loads(EXTRA_AGENTS) if EXTRA_AGENTS.strip() else []

spark.sql("""CREATE TABLE IF NOT EXISTS aql_agents (
    agent_name STRING, workspace_name STRING, stage STRING,
    question_bank STRING, enabled BOOLEAN, registered_at TIMESTAMP) USING DELTA""")
spark.sql("""CREATE TABLE IF NOT EXISTS aql_ground_truth (
    bank_name STRING, question_id STRING, question STRING, expected_answer STRING,
    category STRING, tags STRING, loaded_at TIMESTAMP) USING DELTA""")
spark.sql("""CREATE TABLE IF NOT EXISTS aql_eval_runs (
    run_id STRING, agent_name STRING, workspace_name STRING, stage STRING,
    bank_name STRING, question_count INT, started_at TIMESTAMP, finished_at TIMESTAMP,
    duration_s DOUBLE, status STRING, error STRING, sdk_version STRING) USING DELTA""")
print("Tables ready.")

In [ ]:
# Build the agent registry (merge: existing rows win, new rows appended) -------
import pandas as pd
from datetime import datetime

rows = []
if AUTO_DISCOVER:
    import sempy.fabric as sf
    ws_name = sf.resolve_workspace_name()
    items = sf.list_items()
    for _, it in items[items["Type"] == "DataAgent"].iterrows():
        rows.append({"agent_name": it["Display Name"], "workspace_name": ws_name,
                     "stage": "production", "question_bank": DEFAULT_BANK, "enabled": True})
for e in EXTRA_AGENTS:
    rows.append({"stage": "production", "question_bank": DEFAULT_BANK, "enabled": True, **e})

new = pd.DataFrame(rows, columns=["agent_name", "workspace_name", "stage", "question_bank", "enabled"])
new = new.drop_duplicates(subset=["agent_name", "workspace_name"])
new["registered_at"] = datetime.utcnow()

try:
    cur = spark.read.table("aql_agents").toPandas()
except Exception:
    cur = pd.DataFrame(columns=new.columns)

if len(cur):
    existing = set(zip(cur["agent_name"], cur["workspace_name"]))
    add = new[[ (a, w) not in existing for a, w in zip(new["agent_name"], new["workspace_name"]) ]]
    registry = pd.concat([cur, add], ignore_index=True)
else:
    registry = new

if len(registry):
    spark.createDataFrame(registry).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("aql_agents")
print(f"Registry: {len(registry)} agent(s)")
display(registry)

In [ ]:
# Load & validate ground-truth banks from Files/ground_truth/*.csv ------------
import hashlib, os

GT_DIR = "/lakehouse/default/Files/ground_truth"
os.makedirs(GT_DIR, exist_ok=True)

template = os.path.join(GT_DIR, "_template.csv")
if not os.path.exists(template):
    with open(template, "w", encoding="utf-8") as f:
        f.write('question,expected_answer,category,tags\n"How many rows are in table X?","42","counts",""\n')

banks, problems = [], []
for fn in sorted(os.listdir(GT_DIR)):
    if not fn.endswith(".csv") or fn.startswith("_"):
        continue
    df = pd.read_csv(os.path.join(GT_DIR, fn))
    missing = {"question", "expected_answer"} - set(df.columns)
    if missing:
        problems.append(f"{fn}: missing columns {sorted(missing)}"); continue
    n0 = len(df)
    df = df.dropna(subset=["question", "expected_answer"])
    df["question"] = df["question"].astype(str).str.strip()
    df["expected_answer"] = df["expected_answer"].astype(str).str.strip()
    df = df[(df["question"] != "") & (df["expected_answer"] != "")].drop_duplicates(subset=["question"])
    if len(df) < n0:
        problems.append(f"{fn}: dropped {n0 - len(df)} empty/duplicate row(s)")
    df["bank_name"] = fn[:-4]
    df["question_id"] = df["question"].map(lambda q: hashlib.sha1(q.lower().encode()).hexdigest()[:12])
    for c in ("category", "tags"):
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("").astype(str)
    df["loaded_at"] = datetime.utcnow()
    banks.append(df[["bank_name", "question_id", "question", "expected_answer", "category", "tags", "loaded_at"]])

for p in problems:
    print("WARN:", p)
if banks:
    gt = pd.concat(banks, ignore_index=True)
    spark.createDataFrame(gt).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("aql_ground_truth")
    print(f"Loaded {len(gt)} question(s) across {gt['bank_name'].nunique()} bank(s)")
    display(gt.groupby("bank_name").size().rename("questions").reset_index())
else:
    print("No banks found - drop CSVs into Files/ground_truth/ (see _template.csv) and re-run.")